In [265]:
# Class which can apply different operations to move a molecule around another molecule to generate different conformations of the reactant complex. 
# This is used to generate different conformations of the reactant complex for the reaction prediction model.
import os
import numpy as np
from scipy import optimize
from ase import Atoms
from ase.io import read, write

class OrientMoleculeSphere:
    def __init__(self, molecule1: Atoms, molecule2: Atoms, atom_1: int, atom_2: int):
        self.molecule1_init = molecule1  # The molecule around which the other molecule will be translated (the "anchor" molecule)
        self.molecule2_init = molecule2  # The molecule which will be translated around the anchor molecule
        self.atom_1 = atom_1        # Atom index of molecule 1 
        self.atom_2 = atom_2        # Atom index of molecule 2
        self.count = 0
        
        #self.sphere_center = self.molecule1_init.positions.mean(axis=0) # Center of the sphere is the mean positions of the first molecule
        #self.sphere_radius = np.max(np.linalg.norm(self.molecule1_init.positions - self.sphere_center, axis=1)) - 1# + sphere_radius_buffer # Adding a buffer.
        
        self.sphere_center = self.molecule1_init[self.atom_1].position # Center of the sphere is the position of the atom of interest in molecule 1
        self.sphere_radius = 5.0

        # Place molecule 2 at a random position on the surface of the sphere around molecule 1
        # set seed
        #np.random.seed(42)
        random_theta = np.random.uniform(0, 2 * np.pi)
        random_phi = np.random.uniform(0, np.pi)
        random_x = self.sphere_radius * np.sin(random_phi) * np.cos(random_theta)
        random_y = self.sphere_radius * np.sin(random_phi) * np.sin(random_theta)
        random_z = self.sphere_radius * np.cos(random_phi)
        random_position = np.array([random_x, random_y, random_z]) + self.sphere_center
        translation_vector = random_position - self.molecule2_init[self.atom_2].position
        for atom in self.molecule2_init:
            atom.position += translation_vector
        # Now for the movable molecules
        self.molecule1 = self.molecule1_init.copy()
        self.molecule2 = self.molecule2_init.copy()
    # Utility functions
    def normalize_vector(self, vector):
        return vector / np.linalg.norm(vector)
    
    def rotation_matrix_from_rotvec(self, rotvec):
        theta = np.linalg.norm(rotvec)
        if theta < 1e-12:
            return np.eye(3)
        axis = rotvec / theta
        ux, uy, uz = axis

        # skew-symmetric matrix
        K = np.array([[0, -uz, uy],
                      [uz, 0, -ux],
                      [-uy, ux, 0]])
        
        # Rodrigues' rotation formula
        R = np.eye(3) + np.sin(theta) * K + (1 - np.cos(theta)) * (K @ K)
        return R
    
    # Translation and rotation on sphere
    def step_on_sphere(self, point_on_sphere: list, center: list, step_theta: float, step_phi: float):
        """Translate a point on the surface of a sphere by given angles theta and phi. Units of angles is radians."""
        
        x, y, z = point_on_sphere
        a, b, c = center

        r = self.sphere_radius # Radius of the sphere
        theta = np.arccos((z - c) / r) - step_theta
        phi = np.arctan((y - b) / (x - a)) - step_phi

        x_new = a + r * np.sin(theta) * np.cos(phi)
        y_new = b + r * np.sin(theta) * np.sin(phi)
        z_new = c + r * np.cos(theta)

        return [x_new, y_new, z_new]
    
    def translate_on_sphere(self, molecule: Atoms, initial_point: list, center: list, step_theta: float, step_phi: float):
        """Translate the molecule by moving it along the surface of a sphere."""
        new_position = self.step_on_sphere(initial_point, center, step_theta, step_phi)
        translation_vector = np.array(new_position) - np.array(initial_point)
        
        # Translate the molecule by the calculated translation vector
        for atom in molecule:
            atom.position += translation_vector

    def is_within_sphere(self, point: list, center: list, radius: float):
        """Check if a point is within a sphere defined by its center and radius."""
        return np.linalg.norm(np.array(point) - np.array(center)) <= radius

    def drone_translate(self, step_theta: float, step_phi: float, rotvec: np.ndarray):
        """Translate molecule2 around molecule1 by moving it along the surface of a sphere and applying rotations."""
        # Apply translation on the sphere
        self.translate_on_sphere(self.molecule2, self.molecule2[self.atom_2].position, self.sphere_center, step_theta, step_phi)

        # Apply rotation
        rotation = self.rotation_matrix_from_rotvec(rotvec)
        anchor = self.molecule2[self.atom_2].position
        self.molecule2.positions = (self.molecule2.positions - anchor) @ rotation.T + anchor

    def omega_constraint(self, coordsA: np.ndarray, coordsB: np.ndarray, indexA: int, indexB:int):
        # Make the two masks for the two molecules
        maskA = np.ones(len(coordsA), dtype=bool)
        maskB = np.ones(len(coordsB), dtype=bool)
        maskA[indexA] = False
        maskB[indexB] = False

        r_A = (coordsA[indexA] - coordsA[maskA])
        r_B = (coordsB[indexB] - coordsB[maskB])
        omegas = []
        for vecA in r_A:
            for vecB in r_B:
                omega = np.arccos(np.dot(vecA, vecB) / (np.linalg.norm(vecA) * np.linalg.norm(vecB)))
                omegas.append(omega) 
        return omegas

    def opt_scipy_2(self):

        def objective_function_translate(Vars, p: float = 12.0):
            step_theta, step_phi = Vars

            ## Define position of molecule 2 based on the current step on the sphere
            #coords = self.step_on_sphere(self.molecule2[self.atom_2].position, self.sphere_center, step_theta, step_phi)
            ## Calculate distances from all atoms in molecule 1 to the new position of molecule 2
            #x = np.array(coords)  # Position of the atom of interest in molecule 2 after translation
            #distances = self.molecule1.positions - x
            ## We do the same with all atoms in molecule 2 to the atom of interest in molecule 1
            #y = self.molecule1[self.atom_1].position  # Position of the atom of interest in molecule 1
#
            #distances = np.concatenate((distances, self.molecule2.positions - y), axis=0)
            #
            #
#
            #score = np.sum(1.0 / distances**p + 1.0 / distances**(p/2))  # Inverse of squared distances to penalize closer contacts more heavily
#
            #return score

            # Apply translation on the sphere
            coords = self.molecule2.copy()
            self.translate_on_sphere(coords, coords[self.atom_2].position, self.sphere_center, step_theta, step_phi)

            # Let us reduce steric hindrence between the atom if interest in molecule 2 and all atoms in molecule 1:
            distances = np.linalg.norm(coords.positions - self.molecule1[self.atom_1].position, axis=1)

            # Dont do the same for the other molecule, as we will rotate later
            min_distance = np.min(distances)
            
            # Punish closer contacts more heavily by using a high exponent
            val = 1.0 / (min_distance**p + min_distance**(p/2))  # Inverse of squared distances to penalize closer contacts more heavily
            return val


        # Bounds for the decision variables
        upper_bound_translate = [ np.pi*2,  np.pi*2]  
        lower_bound_translate = [-np.pi*2, -np.pi*2]
        bounds_translate = optimize.Bounds(lower_bound_translate, upper_bound_translate)

        # Initial guess
        initial_guess_translate = [0.0, 0.0]
        # Optimize translation first
        result_translate = optimize.differential_evolution(objective_function_translate,
                                        bounds=bounds_translate,
                                        strategy='best1bin',
                                        maxiter=100,
                                        tol=1e-6)
        
        return result_translate

    def rotate_around_anchor(self, molecule: Atoms, rot_vec: np.ndarray, anchor_index: int):
        # Make items mutable
        coords = molecule.copy()
        positions = coords.positions.copy()
        anchor = positions[anchor_index].copy()

        # Translate to origin
        positions -= anchor

        # Apply rotation
        # Create rotation matrix from rotation vector
        rotation = self.rotation_matrix_from_rotvec(rot_vec)
        positions = positions @ rotation.T
        # Translate back
        positions += anchor
        coords.positions = positions

        return coords
        
    def distance_objective(self, rotvec: np.ndarray, molecule, anchor_index: int, target_point: np.ndarray, p = 6.0):
        rotated = self.rotate_around_anchor(molecule, rotvec, anchor_index)
        # Exclude anchor atom
        mask = np.ones(len(molecule), dtype=bool)
        mask[anchor_index] = False
    
        coords_to_evaluate = rotated.positions[mask]
        # Lennard-Jones-like objective to maximize minimum distance
        distances = np.linalg.norm(coords_to_evaluate - target_point, axis=1)**p
        # we want to maximize min distance
        
        val = 1 * np.min(distances) 
        return val

    def optimize_rotation(self, molecule: Atoms, anchor_index: int, target_point: np.ndarray, p: float = 6.0):
        initial_rotvec = np.array([0.0, 0.0, 0.0])  # Start with no rotation
        result = optimize.minimize(self.distance_objective,
                                   args=(molecule, anchor_index, target_point, p),
                                   x0=initial_rotvec, method='L-BFGS-B', options={'maxiter': 100, 'ftol': 1e-6})
        return result.x


In [266]:
# Example usage
molecule1 = read('molecule1.xyz')  # Replace with actual file path
molecule2 = read('molecule2.xyz')  # Replace with actual file path

pair_of_atoms = [(8, 1), (23, 1), (1, 1), (13, 1), (4, 1), (9, 1), (24, 1), (16, 1), (20, 1), (26, 1), (18, 1), (22, 1)]
traj_file = 'optimization_trajectory.xyz'
# Remove existing trajectory file if it exists

if os.path.exists(traj_file):
    os.remove(traj_file)
for atom1_index, atom2_index in pair_of_atoms:
    # Convert to 0-based index
    atom1_index -= 1 
    atom2_index -= 1

    # Define the orientor object for this pair of atoms
    orientor = OrientMoleculeSphere(molecule1, molecule2, atom1_index, atom2_index)

    # Optimize translation
    result_translate= orientor.opt_scipy_2()
    best_step_theta, best_step_phi = result_translate.x

    # Apply translation on the sphere
    orientor.step_on_sphere(orientor.molecule2[orientor.atom_2].position, orientor.molecule1[orientor.atom_1].position, best_step_theta, best_step_phi)

    # Optimize rotation
    #opt_rot = orientor.optimize_rotation(orientor.molecule1, orientor.atom_2, orientor.molecule1[orientor.atom_2].position)
    #orientor.molecule1.positions = orientor.rotate_around_anchor(orientor.molecule1, opt_rot, orientor.atom_1).positions

    # Merge the two molecules and write to file
    merged_molecule = orientor.molecule1 + orientor.molecule2
    write(f'optimized_merged_molecule_{atom1_index}_{atom2_index}.xyz', merged_molecule,)
    
    # Append to trajectory file
    with open(traj_file, 'a') as traj:
        traj.write(f"{len(merged_molecule)}\n")
        traj.write(f"Optimized merged molecule for atom pair ({atom1_index}, {atom2_index})\n")
        for atom in merged_molecule:
            traj.write(f"{atom.symbol} {atom.position[0]} {atom.position[1]} {atom.position[2]}\n")
    print('Next')

#traj_file2 = 'conf.xyz'
## Remove existing trajectory file if it exists
#if os.path.exists(traj_file2):
#    os.remove(traj_file2)
#
#atom_pairs = [(3, 2), (3, 11), (3, 5), (3, 15), (3, 6), (3, 14), (3, 3), (3, 10), (3, 25), (3, 7), (3, 17), (3, 12), (3, 19)]
#molecule1 = read('molecule2.xyz')
#molecule2 = read('molecule1.xyz')
#for atom1_index, atom2_index in atom_pairs:
#    # Convert to 0-based index
#    atom1_index -= 1
#    atom2_index -= 1
#
#    orientor = OrientMoleculeSphere(molecule1, molecule2, atom1_index, atom2_index)
#
#    result_translate= orientor.opt_scipy_2()
#    best_step_theta, best_step_phi = result_translate.x
#
#    orientor.step_on_sphere(orientor.molecule2[orientor.atom_2].position, orientor.molecule1[orientor.atom_1].position, best_step_theta, best_step_phi)
#
#    #opt_rot = orientor.optimize_rotation(orientor.molecule1, orientor.atom_1, orientor.molecule2[orientor.atom_2].position)
#    #orientor.molecule1.positions = orientor.rotate_around_anchor(orientor.molecule1, opt_rot, orientor.atom_1).positions
#
#    merged_molecule = orientor.molecule1 + orientor.molecule2
#    write(f'optimized_merged_molecule_{atom1_index}_{atom2_index}.xyz', merged_molecule,)
#    with open(traj_file2, 'a') as traj:
#        traj.write(f"{len(merged_molecule)}\n")
#        traj.write(f"Optimized merged molecule for atom pair ({atom1_index}, {atom2_index})\n")
#        for atom in merged_molecule:
#            traj.write(f"{atom.symbol} {atom.position[0]} {atom.position[1]} {atom.position[2]}\n")
#    print('Next')

Next
Next
Next
Next
Next
Next
Next
Next
Next
Next
Next
Next
